In [3]:
%pip install --quiet -U langchain_aws langchain_core langgraph
%pip install --upgrade python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] =os.getenv('AWS_DEFAULT_REGION')

In [5]:
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_PROJECT"] = "day-3-lab1"

In [6]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, AnyMessage

prepared message for conversations

In [7]:
messages = [AIMessage(content=f"So you said you were researching ocean mammals?", name="Model")]
messages.append(HumanMessage(content=f"Yes, that's right.",name="Vijay"))
messages.append(AIMessage(content=f"Great, what would you like to learn about.", name="Model"))
messages.append(HumanMessage(content=f"I want to learn about the best place to see Orcas in the India.", name="Vijay"))

for m in messages:
    m.pretty_print()

================================== Ai Message ==================================
Name: Model

So you said you were researching ocean mammals?
================================ Human Message =================================
Name: Vijay

Yes, that's right.
================================== Ai Message ==================================
Name: Model

Great, what would you like to learn about.
================================ Human Message =================================
Name: Vijay

I want to learn about the best place to see Orcas in the India.


Configure the LLM model  for conversation

In [8]:
from langchain_aws import ChatBedrock
# --- Chat Model ---
llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    temperature=1,
)
result = llm.invoke("hello")
print(result)

content="Hello! How can I help you today? If you have any questions or topics you'd like to discuss, feel free to let me know. Whether it's about science, technology, history, or anything else, I'm here to provide information and assist you. If you're looking for recommendations or advice on a specific subject, just ask, and I'll do my best to help you out. Happy to be of service! 😊🌟" additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '7f3ae844-29c4-4ef5-8310-70254c085d1b', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Fri, 13 Mar 2026 05:03:32 GMT', 'content-type': 'application/json', 'content-length': '584', 'connection': 'keep-alive', 'x-amzn-requestid': '7f3ae844-29c4-4ef5-8310-70254c085d1b'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [656]}, 'model_provider': 'bedrock_converse', 'model_name': 'amazon.nova-lite-v1:0'} id='lc_run--019ce594-03dd-7ac0-ad57-4cb51e601971-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_

In [9]:
response = llm.invoke(messages)
print(response)

ValidationException: An error occurred (ValidationException) when calling the Converse operation: A conversation must start with a user message. Try again with a conversation that starts with a user message.

Define State with messages

from typing_extensions import TypedDict

class MessageState(TypedDict):
    messages: list[AnyMessage]

Reducers- where atutomatically append all the message 

In [12]:
from typing import  List,  TypedDict, Annotated,operator, Literal

from typing_extensions import TypedDict
from langgraph.graph import MessagesState # messages[data memeber] add_messages [method used for adding the messages]

class MessageState(MessagesState):
    pass
    #messages: Annotated[list[AnyMessage], operator.add]

In [13]:
from langgraph.graph.message import add_messages

initial_messages = [
    HumanMessage(content="hi", name="vijay"),
    AIMessage(content="hello! how can i assist you")
]
new_prompt = "what is the best place to see orcas in india?"

update_message = add_messages(initial_messages, new_prompt)

print(update_message)

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, name='vijay', id='ac09f77c-5e72-4c78-b799-356b7174d9cf'), AIMessage(content='hello! how can i assist you', additional_kwargs={}, response_metadata={}, id='f53708dd-c7b2-4adf-bb77-a69b7ed61bbe', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is the best place to see orcas in india?', additional_kwargs={}, response_metadata={}, id='22d09432-45f4-402a-972c-544bd30ac794')]


Tools: for fetching addtional context for your prompt you can use a tool 
        or 
       do an action on external system [servicenow [ticket]]

In [14]:

#tool1 multiplication of two numbers
def multiply(a: int,b: int) -> int:
    """ Multiply a and b.
    Args:
        a: int
        b: int
    """
    return a * b

# tool2 add of 2 numbers 
def addition(a: int,b: int) -> int:
    """ add a and b.
    Args:
        a: int
        b: int
    """
    return a + b 

llm_with_tools = llm.bind_tools([multiply, addition])

In [15]:
result = llm_with_tools.invoke("what is the multiplication of 5 and 10")

In [ ]:
print(result.message.content)

content=[{'type': 'text', 'text': "<thinking>The User is asking for the multiplication of two numbers, 5 and 10. I have the tool 'multiply' that can perform this task. I need to pass these two numbers as arguments to the 'multiply' tool.</thinking>\n"}, {'type': 'tool_use', 'name': 'multiply', 'input': {'a': 5, 'b': 10}, 'id': 'tooluse_bJL6v61jkTi65lRj8hNe68'}] additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': 'a7821826-1deb-401f-a79c-adef25b425d3', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Fri, 13 Mar 2026 05:08:42 GMT', 'content-type': 'application/json', 'content-length': '519', 'connection': 'keep-alive', 'x-amzn-requestid': 'a7821826-1deb-401f-a79c-adef25b425d3'}, 'RetryAttempts': 1}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [772]}, 'model_provider': 'bedrock_converse', 'model_name': 'amazon.nova-lite-v1:0'} id='lc_run--019ce597-d88b-78d3-aabe-ca3e093c51bd-0' tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 10}, 'id': 'tooluse_bJL6v61j